In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# 1. 스파크 세션 생성 (로컬 테스트용, 복잡한 외부 패키지 불필요)
spark = (
    SparkSession.builder.appName("ExtremeIngestionLocalTest")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")  # AQE 활성화
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.files.maxPartitionBytes", "134217728")  # 128MB 파티션
    .getOrCreate()
)

# 2. 테스트용 가짜 데이터(CSV)를 만들기 위한 경로 설정
local_dir = "./test_data"
os.makedirs(local_dir, exist_ok=True)
sample_csv_path = os.path.join(local_dir, "sample_logs.csv")

# 가상의 CSV 파일 생성 (실무에서는 이 위치에 수집할 파일들이 쌓임)
with open(sample_csv_path, "w", encoding="utf-8") as f:
    f.write("log_id,user_id,action_type,amount,event_time\n")
    f.write("LOG-001,101,click,15.5,2026-06-01 10:00:00\n")
    f.write("LOG-002,102,purchase,120.0,2026-06-01 11:30:00\n")
    f.write("LOG-003,101,view,5.0,2026-06-02 09:15:00\n")

# 3. [필수] 스키마 사전 정의 (inferSchema 차단으로 속도 극대화)
ingest_schema = StructType(
    [
        StructField("log_id", StringType(), False),
        StructField("user_id", IntegerType(), True),
        StructField("action_type", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("event_time", TimestampType(), True),
    ]
)

# 4. [고성능 수집 단계]
df_raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "false")  # 스키마 추론 생략 (병목 차단)
    .schema(ingest_schema)
    .csv(local_dir)
)

print("--- [수집된 원본 데이터 확인] ---")
df_raw.show()

# 5. [가공 및 적재 최적화]
# 날짜별 파티션을 위한 컬럼 생성
df_refined = df_raw.withColumn(
    "event_date", df_raw["event_time"].cast("date")
)

# 6. [최종 저장] Parquet 포맷 + 날짜별 파티셔닝 적용 적재
target_output_path = "./optimized_output_logs"

(
    df_refined.write.mode("overwrite")
    .partitionBy("event_date")
    .option("compression", "snappy")
    .parquet(target_output_path)
)

print(
    f"대용량 수집 및 최적화 Parquet 적재 완료! 저장 경로: {target_output_path}"
)

--- [수집된 원본 데이터 확인] ---
+-------+-------+-----------+------+-------------------+
| log_id|user_id|action_type|amount|         event_time|
+-------+-------+-----------+------+-------------------+
|LOG-001|    101|      click|  15.5|2026-06-01 10:00:00|
|LOG-002|    102|   purchase| 120.0|2026-06-01 11:30:00|
|LOG-003|    101|       view|   5.0|2026-06-02 09:15:00|
+-------+-------+-----------+------+-------------------+

대용량 수집 및 최적화 Parquet 적재 완료! 저장 경로: ./optimized_output_logs


In [5]:
import pandas as pd
import numpy as np
import time
import os

# 1. 2만 행의 데이터 생성
data = {
    'id': range(50000),
    'category': np.random.choice(['A', 'B', 'C', 'D'], 50000),
    'value': np.random.randn(50000),
    'timestamp': pd.date_range(start='2026-01-01', periods=50000, freq='min')
}
df = pd.DataFrame(data)

# 2. 성능 측정 준비
output_dir = "performance_test"
os.makedirs(output_dir, exist_ok=True)

# 3. CSV 저장 테스트
start_csv = time.time()
df.to_csv(os.path.join(output_dir, "test_data.csv"), index=False)
end_csv = time.time()
csv_duration = end_csv - start_csv
csv_size = os.path.getsize(os.path.join(output_dir, "test_data.csv"))

# 4. Parquet 저장 테스트
start_parquet = time.time()
df.to_parquet(os.path.join(output_dir, "test_data.parquet"), index=False, compression='snappy')
end_parquet = time.time()
parquet_duration = end_parquet - start_parquet
parquet_size = os.path.getsize(os.path.join(output_dir, "test_data.parquet"))

# 5. 결과 요약
results = {
    "Format": ["CSV", "Parquet"],
    "Time (seconds)": [csv_duration, parquet_duration],
    "File Size (bytes)": [csv_size, parquet_size]
}
results_df = pd.DataFrame(results)
print(results_df)

    Format  Time (seconds)  File Size (bytes)
0      CSV        0.094173            2370677
1  Parquet        0.018876            1217028


In [3]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
from pyspark.sql import SparkSession

# 1. 스파크 세션 생성
spark = (
    SparkSession.builder.appName("MelonChartAnalysis")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

# 2. 멜론 TOP 100 크롤링 (Requests + BeautifulSoup)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
url = "https://www.melon.com/chart/index.htm"
res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

rank_list, title_list, artist_list = [], [], []

# 50위까지, 100위까지의 태그 선택자 파싱
for item in soup.select("tbody > tr"):
    try:
        rank = item.select_one("span.rank").text
        title = item.select_one("div.rank01 a").text
        artist = item.select_one("div.rank02 a").text

        rank_list.append(int(rank))
        title_list.append(title)
        artist_list.append(artist)
    except AttributeError:
        continue

# 판다스 DataFrame 생성 후 스파크 DataFrame으로 변환
pd_df = pd.DataFrame(
    {"rank": rank_list, "title": title_list, "artist": artist_list}
)
df_melon = spark.createDataFrame(pd_df)

# [핵심 수집] 즉시 Parquet 포맷으로 압축 저장 (작은 파일 방지 및 메타데이터 확보)
df_melon.write.mode("overwrite").parquet("./melon_top100.parquet")
print("멜론 TOP 100 수집 및 Parquet 적재 완료!")

멜론 TOP 100 수집 및 Parquet 적재 완료!


In [19]:
from pyspark.sql import functions as F

# 1. 저장해 둔 파켓(Parquet) 파일 읽어오기 (컬럼 프로젝션과 메타데이터 즉시 로드)
df_melon = spark.read.parquet("./melon_top100.parquet")

print("--- [1. 수집된 데이터 확인] ---")
df_melon.show(10, truncate=False)

# -------------------------------------------------------------------------
# 2. 집계(Aggregation) 성능 최적화 예시: 아티스트별 곡 수 집계
# -------------------------------------------------------------------------
# 스파크는 셔플 전에 각 노드에서 미리 집계(Map-side Aggregation)를 수행합니다.
print("--- [2. 아티스트별 TOP 100 수록 곡 수 집계] ---")
artist_counts = (
    df_melon.groupBy("artist")
    .agg(
        F.count("title").alias("song_count"),
        F.min("rank").alias("highest_rank"),  # 가장 높은 순위
    )
    .orderBy(F.desc("song_count"), F.asc("highest_rank"))
)

artist_counts.show(20)

# -------------------------------------------------------------------------
# 3. 조인(Join) 성능 극대화 예시: 브로드캐스트 조인(Broadcast Join) 활용
# -------------------------------------------------------------------------
# 만약 가상의 '기획사(Agency)' 정보가 담긴 작은 데이터프레임이 있다고 가정해 봅시다.
agency_data = [
    ("aespa", "SM Entertainment"),
    ("IVE (아이브)", "Starship Entertainment"),
    ("RESCENE (리센느)", "더뮤즈 엔터테인먼트"),
    # ... (실무에서는 이 테이블이 수십만 건 이상일 수 있음)
]
df_agency = spark.createDataFrame(agency_data, ["artist", "agency"])

df_melon_clean = df_melon.withColumn(
    "artist", F.trim(F.col("artist"))
)  # 앞뒤 공백 제거

# [핵심] 작은 테이블은 broadcast() 함수로 감싸주어 네트워크 셔플을 원천 차단합니다.
print("--- [3. 브로드캐스트 조인을 통한 기획사 매핑] ---")
df_joined = df_melon_clean.join(
    F.broadcast(df_agency), on="artist", how="left"  # 작은 테이블 브로드캐스트 강제
)

df_joined.select("rank", "title", "artist", "agency").show(30, truncate=False)

# 곡의 개수 집계까지는 완벽하나
# 브로드캐스트 조인에서 다소 아쉬운 결과가 나왔다

--- [1. 수집된 데이터 확인] ---
+----+-----------+------------------+
|rank|title      |artist            |
+----+-----------+------------------+
|1   |BiiiG      |BIGBANG (빅뱅)    |
|2   |LOVE ATTACK|RESCENE (리센느)  |
|3   |갑자기     |아이오아이 (I.O.I)|
|4   |REDRED     |CORTIS (코르티스) |
|5   |Pretty Girl|RESCENE (리센느)  |
|6   |LEMONADE   |aespa             |
|7   |Deja Vu    |RESCENE (리센느)  |
|8   |만찬가     |태연 (TAEYEON)    |
|9   |It′s Me    |아일릿(ILLIT)     |
|10  |BAD        |ATEEZ(에이티즈)   |
+----+-----------+------------------+
only showing top 10 rows

--- [2. 아티스트별 TOP 100 수록 곡 수 집계] ---
+----------------------+----------+------------+
|                artist|song_count|highest_rank|
+----------------------+----------+------------+
|            방탄소년단|         6|          41|
|      RESCENE (리센느)|         4|           2|
|Hearts2Hearts (하츠...|         4|          16|
|       DAY6 (데이식스)|         4|          23|
|          IVE (아이브)|         4|          30|
|                 aespa|         3|      

In [20]:
# 리센느와 아이브가 포함된 행만 필터링해서 원본 문자열 출력 (앞뒤로 |를 붙여 공백 확인)
df_melon.filter(
    F.col("artist").like("%RESCENE%") | F.col("artist").like("%IVE%")
).select(
    F.concat(F.lit("["), F.col("artist"), F.lit("]"))
).show(
    truncate=False
)

+--------------------+
|concat([, artist, ])|
+--------------------+
|[RESCENE (리센느)]  |
|[RESCENE (리센느)]  |
|[RESCENE (리센느)]  |
|[IVE (아이브)]      |
|[RESCENE (리센느)]  |
|[IVE (아이브)]      |
|[IVE (아이브)]      |
|[IVE (아이브)]      |
+--------------------+



In [21]:
# 예시: 멜론 아티스트 이름에서 대소문자 무시 및 정제 후 조인하기
# (만약 멜론에는 "RESCENE"으로만 나오고, agency_data에는 "RESCENE (리센느)"로 되어있다면 방향을 맞춰주어야 합니다.)

# 1. 멜론 데이터의 아티스트 이름을 대문자로 통일하고 공백 제거
df_melon_clean = df_melon.withColumn(
    "artist_key", F.upper(F.trim(F.col("artist")))
)

# 2. 기획사 데이터도 동일하게 키 값 생성
agency_data = [
    ("AESPA", "SM Entertainment"),
    ("IVE", "Starship Entertainment"),  # 괄호 제거하고 핵심 키워드만 입력
    ("RESCENE", "더뮤즈 엔터테인먼트"),
]
df_agency = spark.createDataFrame(agency_data, ["artist_key", "agency"])

# 3. 정제된 키(artist_key)를 기준으로 브로드캐스트 조인 수행
df_joined = df_melon_clean.join(
    F.broadcast(df_agency), on="artist_key", how="left"
)

# 결과 확인 (artist_key 대신 원래 artist를 보여주도록 선택)
df_joined.select(
    "rank", "title", F.col("artist").alias("original_artist"), "agency"
).filter(
    F.col("original_artist").like("%RESCENE%")
    | F.col("original_artist").like("%IVE%")
).show(
    30, truncate=False
)

# 여전히 브로드캐스트 조인에서 다소 아쉬운 결과가 나왔다
# 즉 특수 공백 문자의 가능성이 있다

+----+-----------+----------------+------+
|rank|title      |original_artist |agency|
+----+-----------+----------------+------+
|2   |LOVE ATTACK|RESCENE (리센느)|NULL  |
|5   |Pretty Girl|RESCENE (리센느)|NULL  |
|7   |Deja Vu    |RESCENE (리센느)|NULL  |
|30  |BANG BANG  |IVE (아이브)    |NULL  |
|52  |Runaway    |RESCENE (리센느)|NULL  |
|74  |REBEL HEART|IVE (아이브)    |NULL  |
|96  |I AM       |IVE (아이브)    |NULL  |
|100 |BLACKHOLE  |IVE (아이브)    |NULL  |
+----+-----------+----------------+------+



In [22]:
from pyspark.sql import functions as F

# 1. 멜론 데이터의 아티스트 이름에서 특수 공백 문자(\u00A0)나 보이지 않는 공백을 일반 공백으로 치환 후 앞뒤 공백 제거
df_melon_clean = df_melon.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 2. 기획사 데이터도 동일하게 괄호 앞의 특수 공백이나 형태를 맞춰서 정의
agency_data = [
    ("aespa", "SM Entertainment"),
    (
        "IVE (아이브)",
        "Starship Entertainment",
    ),  # 혹은 아래 정규식으로 괄호째 날려버려도 좋습니다.
    ("RESCENE (리센느)", "더뮤즈 엔터테인먼트"),
]

df_agency = spark.createDataFrame(agency_data, ["artist", "agency"])
df_agency_clean = df_agency.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 3. 정제된 키('artist_cleaned')를 기준으로 브로드캐스트 조인 수행
print("--- [브로드캐스트 조인 재시도] ---")
df_joined = df_melon_clean.join(
    F.broadcast(df_agency_clean.select("artist_cleaned", "agency")),
    on="artist_cleaned",
    how="left",
)

# 결과 확인 (원래 아티스트 이름과 매핑된 기획사 확인)
df_joined.select(
    "rank", "title", F.col("artist").alias("original_artist"), "agency"
).filter(
    F.col("original_artist").like("%RESCENE%")
    | F.col("original_artist").like("%IVE%")
).show(
    30, truncate=False
)

--- [브로드캐스트 조인 재시도] ---
+----+-----------+----------------+----------------------+
|rank|title      |original_artist |agency                |
+----+-----------+----------------+----------------------+
|2   |LOVE ATTACK|RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|5   |Pretty Girl|RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|7   |Deja Vu    |RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|30  |BANG BANG  |IVE (아이브)    |Starship Entertainment|
|52  |Runaway    |RESCENE (리센느)|더뮤즈 엔터테인먼트   |
|74  |REBEL HEART|IVE (아이브)    |Starship Entertainment|
|96  |I AM       |IVE (아이브)    |Starship Entertainment|
|100 |BLACKHOLE  |IVE (아이브)    |Starship Entertainment|
+----+-----------+----------------+----------------------+



In [23]:
from pyspark.sql import functions as F

# 1. 기존에 정제해 둔 멜론 데이터프레임 활용 (앞서 만든 artist_cleaned 장착)
df_melon_clean = df_melon.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 2. [확장] 기획사 정보 + 멤버(Members) 정보가 포함된 마스터 데이터 정의
# 실무에서는 이런 데이터가 RDB나 S3의 작은 참조 테이블(Dimension Table)로 존재합니다.
extended_agency_data = [
    ("aespa", "SM Entertainment", "카리나,윈터,지젤,닝닝"),
    ("IVE (아이브)", "Starship Entertainment", "안유진,가을,레이,장원영,리즈,이서"),
    ("RESCENE (리센느)", "더뮤즈 엔터테인먼트", "원이,리브,미나미,제나,메이"),
]

df_master = spark.createDataFrame(
    extended_agency_data, ["artist", "agency", "members"]
)

# 마스터 데이터도 동일하게 공백 정제 키 생성
df_master_clean = df_master.withColumn(
    "artist_cleaned", F.trim(F.regexp_replace(F.col("artist"), "[\u00A0\s]+", " "))
)

# 3. [핵심] 브로드캐스트 조인 수행 (참조 테이블이 작으므로 네트워크 셔플 제로!)
df_fully_joined = df_melon_clean.join(
    F.broadcast(df_master_clean.select("artist_cleaned", "agency", "members")),
    on="artist_cleaned",
    how="left",
)

# 4. 결과 출력 (순위, 제목, 아티스트, 기획사, 멤버 확인)
print("--- [멜론 TOP 100 + 기획사 + 멤버 브로드캐스트 조인 결과] ---")
df_fully_joined.select(
    "rank", 
    "title", 
    F.col("artist").alias("original_artist"), 
    "agency", 
    "members"
).filter(
    F.col("original_artist").like("%RESCENE%") | 
    F.col("original_artist").like("%IVE%") | 
    F.col("original_artist").like("%aespa%")
).show(30, truncate=False)

--- [멜론 TOP 100 + 기획사 + 멤버 브로드캐스트 조인 결과] ---
+----+---------------------------------------------+----------------+----------------------+---------------------------------+
|rank|title                                        |original_artist |agency                |members                          |
+----+---------------------------------------------+----------------+----------------------+---------------------------------+
|2   |LOVE ATTACK                                  |RESCENE (리센느)|더뮤즈 엔터테인먼트   |원이,리브,미나미,제나,메이       |
|5   |Pretty Girl                                  |RESCENE (리센느)|더뮤즈 엔터테인먼트   |원이,리브,미나미,제나,메이       |
|6   |LEMONADE                                     |aespa           |SM Entertainment      |카리나,윈터,지젤,닝닝            |
|7   |Deja Vu                                      |RESCENE (리센느)|더뮤즈 엔터테인먼트   |원이,리브,미나미,제나,메이       |
|30  |BANG BANG                                    |IVE (아이브)    |Starship Entertainment|안유진,가을,레이,장원영,리즈,이서|
|45  |Whiplash                    